In [1]:
import pandas as pd
import sys

# load data set
df = pd.read_csv("../NPFC-Test_Database_V2.csv")

relevant_columns = [
    "Subject_ID",
    "Timestamp",
    "Test_Time",
    "Task_Num",
    "Task_Time",
    "Task_Type",
    "Frame",
    "Task_Frame",
    "Selfreport_valence",
    "Selfreport_arousal",
    "Selfreport_focus",
    "Face_Detection",
    "resmasknet_anger",
    "resmasknet_disgust",
    "resmasknet_fear",
    "resmasknet_happiness",
    "resmasknet_sadness",
    "resmasknet_surprise",
    "resmasknet_neutral",
    "Temperature",
    "EDA",
    "BVP",
    "HeadBandOn",
    "Delta_TP9",
    "Delta_AF7",
    "Delta_AF8",
    "Delta_TP10",
    "Theta_TP9",
    "Theta_AF7",
    "Theta_AF8",
    "Theta_TP10",
    "Alpha_TP9",
    "Alpha_AF7",
    "Alpha_AF8",
    "Alpha_TP10",
    "Beta_TP9",
    "Beta_AF7",
    "Beta_AF8",
    "Beta_TP10",
    "Gamma_TP9",
    "Gamma_AF7",
    "Gamma_AF8",
    "Gamma_TP10",
    "Gender",
    "Perceived_Tiredness",
    "Perceived_Stress",
    "Wearing_Glasses",
]

df = df[relevant_columns]

df.info()


<class 'pandas.DataFrame'>
RangeIndex: 47905 entries, 0 to 47904
Data columns (total 47 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Subject_ID            47905 non-null  str    
 1   Timestamp             47905 non-null  str    
 2   Test_Time             47905 non-null  str    
 3   Task_Num              47905 non-null  float64
 4   Task_Time             47905 non-null  str    
 5   Task_Type             47905 non-null  int64  
 6   Frame                 47905 non-null  int64  
 7   Task_Frame            47905 non-null  int64  
 8   Selfreport_valence    47905 non-null  int64  
 9   Selfreport_arousal    47905 non-null  int64  
 10  Selfreport_focus      47905 non-null  int64  
 11  Face_Detection        47905 non-null  int64  
 12  resmasknet_anger      46606 non-null  float64
 13  resmasknet_disgust    46606 non-null  float64
 14  resmasknet_fear       46606 non-null  float64
 15  resmasknet_happiness  46606 no

In [2]:
sys.path.append("./data-preprocessing")
from data_preparation import (
    prepare_data,
    normalize_signals_with_mediation_baseline,
    align_lag_signals,
)

# Clean, normalize and align data based on exploratory data analysis and domain knowledge
df = normalize_signals_with_mediation_baseline(df)
df = align_lag_signals(df)
df = prepare_data(df)

# Compare number of rows before and after!
df.info()

<class 'pandas.DataFrame'>
Index: 22177 entries, 14 to 47904
Data columns (total 49 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Subject_ID                    22177 non-null  str    
 1   Timestamp                     22177 non-null  str    
 2   Test_Time                     22177 non-null  str    
 3   Task_Num                      22177 non-null  float64
 4   Task_Time                     22177 non-null  str    
 5   Task_Type                     22177 non-null  int64  
 6   Frame                         22177 non-null  int64  
 7   Task_Frame                    22177 non-null  int64  
 8   Selfreport_valence            22177 non-null  int64  
 9   Selfreport_arousal            22177 non-null  int64  
 10  Selfreport_focus              22177 non-null  int64  
 11  Face_Detection                22177 non-null  int64  
 12  resmasknet_anger              22177 non-null  float64
 13  resmasknet_disgu

# Feature selection

Selecting the best features (X).
Also creating the features for prediction (Y).


In [ ]:
from feature_selection import variance_treshold_selection

# Grab resmasknet_dominant_emotion if you also want the emotion
result_df_regression = df["resmasknet_max_emotion_value"]

# Select relevant features
feature_columns = [
    "Selfreport_valence",
    "Selfreport_arousal",
    "Selfreport_focus",
    "Temperature",
    "EDA",
    "BVP",
    "Delta_TP9",
    "Delta_AF7",
    "Delta_AF8",
    "Delta_TP10",
    "Theta_TP9",
    "Theta_AF7",
    "Theta_AF8",
    "Theta_TP10",
    "Alpha_TP9",
    "Alpha_AF7",
    "Alpha_AF8",
    "Alpha_TP10",
    "Beta_TP9",
    "Beta_AF7",
    "Beta_AF8",
    "Beta_TP10",
    "Gamma_TP9",
    "Gamma_AF7",
    "Gamma_AF8",
    "Gamma_TP10",
    "Gender",
    "Perceived_Tiredness",
    "Perceived_Stress",
    "Wearing_Glasses",
]

feature_df = df[feature_columns]
feature_df = variance_treshold_selection(feature_df)

# Output the results of the feature selection
feature_df.info()

<class 'pandas.DataFrame'>
Index: 22177 entries, 14 to 47904
Data columns (total 30 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Selfreport_valence   22177 non-null  int64  
 1   Selfreport_arousal   22177 non-null  int64  
 2   Selfreport_focus     22177 non-null  int64  
 3   Temperature          21902 non-null  float64
 4   EDA                  21902 non-null  float64
 5   BVP                  21902 non-null  float64
 6   Delta_TP9            22177 non-null  float64
 7   Delta_AF7            22177 non-null  float64
 8   Delta_AF8            22177 non-null  float64
 9   Delta_TP10           22177 non-null  float64
 10  Theta_TP9            22177 non-null  float64
 11  Theta_AF7            22177 non-null  float64
 12  Theta_AF8            22177 non-null  float64
 13  Theta_TP10           22177 non-null  float64
 14  Alpha_TP9            22177 non-null  float64
 15  Alpha_AF7            22177 non-null  float64
 16  A

# Model Training


In [ ]:
### Data normalization preprocessor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, MinMaxScaler

# All physiological signals, brainwave bands, and temperature
sensor_and_eeg_features = [
    "Temperature",
    "EDA",
    "BVP",
    "Delta_TP9",
    "Delta_AF7",
    "Delta_AF8",
    "Delta_TP10",
    "Theta_TP9",
    "Theta_AF7",
    "Theta_AF8",
    "Theta_TP10",
    "Alpha_TP9",
    "Alpha_AF7",
    "Alpha_AF8",
    "Alpha_TP10",
    "Beta_TP9",
    "Beta_AF7",
    "Beta_AF8",
    "Beta_TP10",
    "Gamma_TP9",
    "Gamma_AF7",
    "Gamma_AF8",
    "Gamma_TP10",
]

# Subjective human ratings
survey_features = [
    "Selfreport_valence",
    "Selfreport_arousal",
    "Selfreport_focus",
    "Perceived_Tiredness",
    "Perceived_Stress",
]

# Metadata/Demographics left untouched
binary_features = ["Gender", "Wearing_Glasses"]

# Robust pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("robust_sensors", RobustScaler(), sensor_and_eeg_features),
        ("minmax_survey", MinMaxScaler(), survey_features),
        ("pass_binary", "passthrough", binary_features),
    ]
)

# HOW TO USE:
# X_train_scaled = preprocessor.fit_transform(X_train)
# X_test_scaled = preprocessor.transform(X_test)

## Regression models

Here we are tranining the regression models. That is, models that will predict the uncertainty of the resmasknet FER model.


In [4]:
# TODO: Actually train the models here and output the results
# sth like train_NN_model(df) or, if short enough, put it here directly and output results
# Make sure to use the backward elimination method
# like (https://pmc.ncbi.nlm.nih.gov/articles/PMC7349550/#:~:text=Notwithstanding%2C%20not,process)

# I propose making a random forest and a NN model. Both as a classfier and a regressor. So 4 models in total.
# TODO is define the classification treshold. Is 70% certain predition? 80%? 90%? Probably we can test this.

## Classification models

Here we are tranining the classification models. That is, models that will classify if the predition of the resmasknet model with high or low confidence based on the selected features.

The way we define "high or low confidence resmasknet prediction" is by using a treshold. If the resmasknet model predicts and emotion with **_80%_**^[change this to the final number used? Maybe test some of them] confidence or more, then it is a high confidence prediction; otherwise, it is a low confidence one.
